# 01. 텍스트로 이미지 생성하기 — Stable Diffusion

**Stable Diffusion** 은 텍스트 설명(prompt)으로부터 이미지를 생성하는 확산(diffusion) 모델입니다.
Hugging Face의 `diffusers` 라이브러리로 단 몇 줄에 이미지를 생성할 수 있습니다.

1. 파이프라인 불러오기
2. 텍스트로 이미지 생성
3. 여러 프롬프트로 생성 비교

> 모델은 처음 실행 시 Hugging Face Hub에서 자동 다운로드되어 `/workspace/.cache/huggingface`에 저장됩니다.
> 이미지 생성은 GPU 메모리를 많이 사용합니다(VRAM 16GB 이상 권장).

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 1. 파이프라인 불러오기

Stable Diffusion v1.5 파이프라인을 불러옵니다. `torch_dtype=torch.float16`으로 메모리를 절약합니다.
(이 컨테이너는 xformers 대신 PyTorch 내장 어텐션(SDPA)을 사용합니다. 메모리가 빠듯하면 `enable_attention_slicing()`을 켜세요.)

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5',
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    variant='fp16' if device == 'cuda' else None,
    use_safetensors=True,
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()   # 메모리 절약
print('파이프라인 준비 완료')

## 2. 텍스트로 이미지 생성

프롬프트를 주고 이미지를 생성합니다. `seed`를 고정하면 같은 결과를 재현할 수 있습니다.

In [ ]:
prompt = 'a photograph of an astronaut riding a horse on the moon, highly detailed'
generator = torch.Generator(device=device).manual_seed(42)

image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=generator).images[0]

plt.figure(figsize=(6, 6))
plt.imshow(image); plt.axis('off')
plt.title('생성 결과')
plt.show()

## 3. 여러 프롬프트 비교

서로 다른 프롬프트로 생성한 이미지를 나란히 봅니다. 프롬프트가 결과를 어떻게 좌우하는지 확인합니다.

In [ ]:
prompts = [
    'a cute corgi puppy, watercolor painting',
    'a futuristic city at sunset, cyberpunk style',
    'a bowl of ramen, food photography, studio lighting',
]
images = []
for p in prompts:
    g = torch.Generator(device=device).manual_seed(0)
    images.append(pipe(p, num_inference_steps=30, guidance_scale=7.5, generator=g).images[0])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, p in zip(axes, images, prompts):
    ax.imshow(img); ax.axis('off'); ax.set_title(p[:30], fontsize=9)
plt.tight_layout(); plt.show()

### 정리

- `diffusers`의 StableDiffusionPipeline으로 텍스트 프롬프트에서 이미지를 생성했습니다.
- `num_inference_steps`, `guidance_scale`, `seed`가 생성에 영향을 줍니다.
- 다음 노트북에서는 이 파라미터들을 체계적으로 실험합니다.